<img src="https://www.plus2net.com/images/top2.jpg" alt="plus2net logo" width="400">

# Python MySQL Connector UPDATE, executemany(), & rowcount Practice in Google Colab

**Main Tutorial Link:** [Python MySQL Update Tutorial](https://www.plus2net.com/python/mysql-update.php)

---

## 1. Environment Setup & Local MySQL/MariaDB Server Daemon

Install MariaDB server and initialize the local daemon within the Google Colab environment.

In [ ]:
# Stop and clean up any previously running MariaDB/MySQL instances completely
!pkill -9 mariadbd || true
!pkill -9 mysqld || true
!pkill -9 mysqld_safe || true
!rm -rf /var/run/mysqld/* /var/lib/mysql/* /run/mysqld/* || true

# Install MariaDB server and pymysql driver
!apt-get update && apt-get install -y mariadb-server > /dev/null
!pip install pymysql cryptography > /dev/null

# Ensure run directories exist with proper system permissions
!mkdir -p /run/mysqld /var/run/mysqld /var/lib/mysql
!chown -R mysql:mysql /run/mysqld /var/run/mysqld /var/lib/mysql
!chmod 777 /run/mysqld /var/run/mysqld

# Initialize default database schema
!mysql_install_db --user=mysql --basedir=/usr --datadir=/var/lib/mysql > /dev/null

# Start MariaDB daemon in the background
import subprocess
import time
import os

print("Starting MariaDB daemon process...")
subprocess.Popen([
    "/usr/sbin/mariadbd",
    "--user=mysql",
    "--pid-file=/run/mysqld/mysqld.pid",
    "--socket=/run/mysqld/mysqld.sock",
    "--datadir=/var/lib/mysql",
    "--skip-networking=0"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Dynamically wait and poll for the socket file to become active
success = False
for i in range(15):
    if os.path.exists("/run/mysqld/mysqld.sock"):
        success = True
        print(f"Database socket found active after {i} seconds!")
        break
    time.sleep(1)

if not success:
    print("Warning: Database socket not found yet. Attempting setup anyway...")

# Create database and configure dedicated credentials via administrative socket connection
!mysql --socket=/run/mysqld/mysqld.sock -u root -e "CREATE DATABASE IF NOT EXISTS my_db; \
  CREATE USER IF NOT EXISTS 'demo_user'@'localhost' IDENTIFIED BY 'demo_password'; \
  GRANT ALL PRIVILEGES ON my_db.* TO 'demo_user'@'localhost'; \
  FLUSH PRIVILEGES;"

print("MySQL/MariaDB server running and user 'demo_user' configured successfully!")

## 2. Download SQL Dump (`student.sql`) & Import Database

Fetch `student.sql` directly from plus2net and import it into the local MySQL database.

In [ ]:
# Download the student.sql dump file into the Colab environment
!wget -q -O student.sql https://www.plus2net.com/python/download/student.sql || echo "Note: Ensure student.sql is hosted at this path"

# Import student.sql into the my_db database
!mysql --socket=/run/mysqld/mysqld.sock -u root my_db < student.sql

print("Database table 'student' created and populated successfully via student.sql!")

## 3. Single Update & Parameterized Queries

Test updating a single record using PyMySQL parameterized syntax and verify affected rows using `rowcount`.

In [ ]:
import pymysql

conn = pymysql.connect(
    user='demo_user',
    password='demo_password',
    database='my_db',
    unix_socket='/run/mysqld/mysqld.sock'
)
cursor = conn.cursor()

try:
    query = "UPDATE student SET class=%s WHERE id=%s"
    my_data = ('Five', 25)
    
    cursor.execute(query, my_data)
    conn.commit()
    print("Rows updated:", cursor.rowcount)
except Exception as e:
    print("Error:", e)
finally:
    conn.close()

## 4. Bulk Updates Using `executemany()`

Perform bulk updates using a sequence of tuples with PyMySQL's <code>executemany()</code> method.

In [ ]:
conn = pymysql.connect(
    user='demo_user',
    password='demo_password',
    database='my_db',
    unix_socket='/run/mysqld/mysqld.sock'
)
cursor = conn.cursor()

try:
    query = "UPDATE student SET class=%s WHERE id=%s"
    my_data = [
        ('Three', 25),
        ('Four', 26),
        ('Seven', 27)
    ]
    
    cursor.executemany(query, my_data)
    conn.commit()
    print("Bulk Update Rows Updated:", cursor.rowcount)
except Exception as e:
    print("Error:", e)
finally:
    conn.close()

---
### Visit [www.plus2net.com](https://www.plus2net.com) for more Python tutorials, code snippets, and data science guides.